In [ ]:
!pip install transformers

In [ ]:
from transformers import ViTForImageClassification, Trainer, TrainingArguments, AutoImageProcessor
from PIL import Image
import torch
import pandas as pd
import os
from torch.utils.data import Dataset, random_split

In [ ]:
model_name = "google/vit-base-patch16-224"
model = ViTForImageClassification.from_pretrained(model_name, num_labels=2,ignore_mismatched_sizes=True)
feature_extractor = AutoImageProcessor.from_pretrained(model_name)

class HouseDataset(Dataset):
    def __init__(self, df, image_folder, feature_extractor):
        self.df = df
        self.image_folder = image_folder
        self.feature_extractor = feature_extractor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        image_path = os.path.join(self.image_folder, self.df.iloc[idx]["image_name"])
        image = Image.open(image_path).convert("RGB")
        inputs = self.feature_extractor(images=image, return_tensors="pt")
        label = torch.tensor(self.df.iloc[idx]["class"], dtype=torch.long)
        return {"pixel_values": inputs["pixel_values"].squeeze(), "labels": label}

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/super-ai-engineer-season-6-individual-hackathon-house-recognition/train.csv")
image_folder = "/content/drive/MyDrive/super-ai-engineer-season-6-individual-hackathon-house-recognition/train/train" # Corrected path
dataset = HouseDataset(df, image_folder, feature_extractor)

In [ ]:
dataset

In [ ]:
train_size = int(0.8 * len(dataset))
eval_size = len(dataset) - train_size
train_dataset, eval_dataset = random_split(dataset, [train_size, eval_size])

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=4,
    weight_decay=0.01,
    report_to="none",
)

In [ ]:
from sklearn.metrics import accuracy_score
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=1)
    acc = accuracy_score(labels, predictions)
    return {"accuracy": acc}

model.to(device) # Explicitly move model to device

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

In [ ]:
output_model_dir = "./my_trained_model"
model.save_pretrained(output_model_dir)
feature_extractor.save_pretrained(output_model_dir)
print(f"Model and feature extractor saved to {output_model_dir}")

In [ ]:
import os
directory_path = '/content/drive/MyDrive/super-ai-engineer-season-6-individual-hackathon-house-recognition/train/'

if os.path.exists(directory_path):
    contents = os.listdir(directory_path)
    print(f"Contents of '{directory_path}':")
    # Display first 10 items if there are many
    if len(contents) > 10:
        for item in contents[:10]:
            print(item)
        print(f"... and {len(contents) - 10} more items.")
    else:
        for item in contents:
            print(item)
else:
    print(f"Directory not found: {directory_path}")

In [ ]:
def predict(images_folder, model, feature_extractor, device):
    model.eval()
    predictions = []
    image_files = [f for f in os.listdir(images_folder) if f.endswith(".jpg") or f.endswith(".png")]

    for image_file in image_files:
        image_path = os.path.join(images_folder, image_file)
        image = Image.open(image_path).convert("RGB")
        inputs = feature_extractor(images=image, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = model(**inputs)
            predicted_label = torch.argmax(outputs.logits, dim=-1).item()
        predictions.append((image_file, predicted_label))

    return predictions

In [ ]:
test_folder = "/content/drive/MyDrive/super-ai-engineer-season-6-individual-hackathon-house-recognition/test/test"
predictions = predict(test_folder, model, feature_extractor,device)

In [ ]:
predictions

In [ ]:
submit = pd.read_csv("/content/drive/MyDrive/super-ai-engineer-season-6-individual-hackathon-house-recognition/sample_submission.csv")

In [ ]:
submit

In [ ]:
ss = pd.DataFrame(predictions)

In [ ]:
ss

In [ ]:
ss = ss.rename(columns={0: "id", 1: "answer"})
ss["id"] = ss["id"].str.replace(".jpg", "")

In [ ]:
submit = submit.set_index("id")
ss = ss.set_index("id")
df_sorted = ss.reindex(submit.index)

In [ ]:
df_sorted

In [ ]:
df_sorted.to_csv("submit_house.csv")